In [1]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False 

In [3]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')

head_df = df.head()

col_labels = []
for col in head_df.columns:
    if col == '竞争对手或同行业企业':
        col_labels.append('竞争对手\n或同行业企业')
    elif col == '咨询顾问，市场分析及中介机构':
        col_labels.append('咨询顾问，\n市场分析及中介机构')
    else:
        col_labels.append(col)

# 绘制为图片
fig, ax = plt.subplots(figsize=(min(18, 2 + 2*len(head_df.columns)), 2 + 0.5*len(head_df)))
ax.axis('off')
tbl = ax.table(
    cellText=head_df.values,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1.4, 1.4)

for col in range(len(head_df.columns)):
    tbl[(0, col)].set_height(0.12)  

plt.tight_layout()
plt.show()

<Figure size 1296x324 with 1 Axes>

In [6]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')

# 选择年份
year = 2023
df_year = df[df['年份'] == year]

# 提取各企业类型数据
large = df_year[df_year['企业规模'] == '大型企业'].iloc[0, 2:]
medium = df_year[df_year['企业规模'] == '中型企业'].iloc[0, 2:]
small = df_year[df_year['企业规模'] == '小型企业'].iloc[0, 2:]
micro = df_year[df_year['企业规模'] == '微型企业'].iloc[0, 2:]

# 构建网络
G = nx.Graph()
enterprise_nodes = ['大型企业', '中型企业', '小型企业', '微型企业']
partners = list(df.columns[2:])

G.add_nodes_from(enterprise_nodes, bipartite=1)
G.add_nodes_from(partners, bipartite=0)

for p in partners:
    G.add_edge('大型企业', p, weight=large[p])
    G.add_edge('中型企业', p, weight=medium[p])
    G.add_edge('小型企业', p, weight=small[p])
    G.add_edge('微型企业', p, weight=micro[p])

# 分组
brown_partners = ['高等学校', '研究机构', '政府部门或行业协会', '集团内其他企业']
black_partners = [
    '供应商', '客户', '竞争对手或同行业企业',
    '咨询顾问，市场分析及中介机构'
]
other_partners = [p for p in partners if p not in brown_partners and p not in black_partners]


pos = {}
# brown_partners左侧
for i, node in enumerate(brown_partners):
    pos[node] = (0, i)
# 企业节点中间
for i, node in enumerate(enterprise_nodes):
    pos[node] = (1, i + 0.5)
# 其他合作对象右侧
for i, node in enumerate(other_partners):
    pos[node] = (2, i)
# black_partners也放右侧（可选：可单独分组）
for i, node in enumerate(black_partners):
    pos[node] = (2, len(other_partners) + i)

edge_weights = [G[u][v]['weight']/5 for u, v in G.edges()]

# 节点颜色
node_colors = []
color_map = {
    '大型企业': "#ea4d4dff",   # 深红
    '中型企业': "#ff9191ff",   # 偏深红
    '小型企业': "#ffbbbbff",   # 浅红
    '微型企业': "#f4dbdb"    # 最浅红
}
for node in G.nodes():
    if node in enterprise_nodes:
        node_colors.append(color_map[node])
    else:
        node_colors.append('lightblue')

# 边颜色
edge_colors = []
for u, v in G.edges():
    if v in brown_partners:
        edge_colors.append('#27408B')
    elif v in black_partners:
        edge_colors.append('black')
    else:
        edge_colors.append('gray')

plt.figure(figsize=(16, 10))
nx.draw(
    G, pos, with_labels=True, node_size=3000, font_size=14,
    width=edge_weights, node_color=node_colors, edge_color=edge_colors
)
plt.title(f'{year}年各类企业合作网络（分组布局）')
plt.axis('off')
plt.show()

<Figure size 1152x720 with 1 Axes>

In [7]:
import pandas as pd
import networkx as nx
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')

years = sorted(df['年份'].unique())
partners = df.columns[2:]

for year in years:
    df_year = df[df['年份'] == year]
    large = df_year[df_year['企业规模'] == '大型企业'].iloc[0, 2:]
    medium = df_year[df_year['企业规模'] == '中型企业'].iloc[0, 2:]
    small = df_year[df_year['企业规模'] == '小型企业'].iloc[0, 2:]
    micro = df_year[df_year['企业规模'] == '微型企业'].iloc[0, 2:]

    G = nx.Graph()
    G.add_node('大型企业', type='企业')
    G.add_node('中型企业', type='企业')
    G.add_node('小型企业', type='企业')
    G.add_node('微型企业', type='企业')

    for p in partners:
        G.add_node(p, type='合作对象')
        G.add_edge('大型企业', p, weight=large[p])
        G.add_edge('中型企业', p, weight=medium[p])
        G.add_edge('小型企业', p, weight=small[p])
        G.add_edge('微型企业', p, weight=micro[p])

    pr = nx.pagerank(G, weight='weight')
    pr_sorted = sorted(pr.items(), key=lambda x: x[1], reverse=True)
    print(f"\n{year}年 PageRank 结果（按影响力排序）:")
    for node, score in pr_sorted:
        print(f"{node}: {score:.4f}")


2016年 PageRank 结果（按影响力排序）:
大型企业: 0.1369
中型企业: 0.1187
微型企业: 0.1158
小型企业: 0.1131
客户: 0.0788
集团内其他企业: 0.0711
供应商: 0.0696
政府部门或行业协会: 0.0675
高等学校: 0.0666
研究机构: 0.0470
其他: 0.0438
竞争对手或同行业企业: 0.0375
咨询顾问，市场分析及中介机构: 0.0338

2017年 PageRank 结果（按影响力排序）:
大型企业: 0.1350
微型企业: 0.1183
中型企业: 0.1181
小型企业: 0.1129
客户: 0.0789
集团内其他企业: 0.0720
供应商: 0.0696
政府部门或行业协会: 0.0678
高等学校: 0.0659
研究机构: 0.0459
其他: 0.0417
竞争对手或同行业企业: 0.0385
咨询顾问，市场分析及中介机构: 0.0352

2018年 PageRank 结果（按影响力排序）:
大型企业: 0.1299
中型企业: 0.1202
微型企业: 0.1185
小型企业: 0.1158
客户: 0.0874
供应商: 0.0794
集团内其他企业: 0.0712
高等学校: 0.0671
政府部门或行业协会: 0.0623
研究机构: 0.0449
竞争对手或同行业企业: 0.0365
其他: 0.0355
咨询顾问，市场分析及中介机构: 0.0313

2019年 PageRank 结果（按影响力排序）:
大型企业: 0.1416
中型企业: 0.1198
小型企业: 0.1134
微型企业: 0.1095
客户: 0.0865
供应商: 0.0820
集团内其他企业: 0.0711
高等学校: 0.0692
政府部门或行业协会: 0.0572
研究机构: 0.0482
竞争对手或同行业企业: 0.0365
其他: 0.0328
咨询顾问，市场分析及中介机构: 0.0321

2020年 PageRank 结果（按影响力排序）:
大型企业: 0.1392
中型企业: 0.1218
小型企业: 0.1127
微型企业: 0.1107
客户: 0.0811
集团内其他企业: 0.0761
供应商: 0.0753
政府部门或行业协会: 0.0657

In [8]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')

years = sorted(df['年份'].unique())
partners = df.columns[2:]

# 用于保存结果
pagerank_dict = {
    '大型企业': [],
    '中型企业': [],
    '小型企业': [],
    '微型企业': []
}

for year in years:
    df_year = df[df['年份'] == year]
    large = df_year[df_year['企业规模'] == '大型企业'].iloc[0, 2:]
    medium = df_year[df_year['企业规模'] == '中型企业'].iloc[0, 2:]
    small = df_year[df_year['企业规模'] == '小型企业'].iloc[0, 2:]
    micro = df_year[df_year['企业规模'] == '微型企业'].iloc[0, 2:]

    G = nx.Graph()
    G.add_node('大型企业', type='企业')
    G.add_node('中型企业', type='企业')
    G.add_node('小型企业', type='企业')
    G.add_node('微型企业', type='企业')

    for p in partners:
        G.add_node(p, type='合作对象')
        G.add_edge('大型企业', p, weight=large[p])
        G.add_edge('中型企业', p, weight=medium[p])
        G.add_edge('小型企业', p, weight=small[p])
        G.add_edge('微型企业', p, weight=micro[p])

    pr = nx.pagerank(G, weight='weight')
    pagerank_dict['大型企业'].append(pr['大型企业'])
    pagerank_dict['中型企业'].append(pr['中型企业'])
    pagerank_dict['小型企业'].append(pr['小型企业'])
    pagerank_dict['微型企业'].append(pr['微型企业'])

# 可视化
plt.figure(figsize=(10,6))

# 添加2018-2019区间浅蓝色底色
plt.axvspan(2018, 2019, color="#bacfdf", alpha=0.7, zorder=0)
plt.axvspan(2022, 2023, color="#bacfdf", alpha=0.7, zorder=0)

plt.plot(years, pagerank_dict['大型企业'], marker='o', label='大型企业', color='pink')
plt.plot(years, pagerank_dict['中型企业'], marker='o', label='中型企业', color='orange')
plt.plot(years, pagerank_dict['小型企业'], marker='o', label='小型企业', color='green')
plt.plot(years, pagerank_dict['微型企业'], marker='o', label='微型企业', color='cyan')
plt.xlabel('年份')
plt.ylabel('PageRank值')
plt.title('各类企业PageRank值随年份变化')
plt.legend()
plt.grid(True)
plt.show()

# 显示2018-2019，2022-2023之间的数据
print("2018-2019年PageRank值：")
for i, year in enumerate(years):
    if 2018 <= year <= 2019:
        print(f"{year}: 大型企业={pagerank_dict['大型企业'][i]:.4f}, 中型企业={pagerank_dict['中型企业'][i]:.4f}, 小型企业={pagerank_dict['小型企业'][i]:.4f}, 微型企业={pagerank_dict['微型企业'][i]:.4f}")

print("\n2022-2023年PageRank值：")
for i, year in enumerate(years):
    if 2022 <= year <= 2023:
        print(f"{year}: 大型企业={pagerank_dict['大型企业'][i]:.4f}, 中型企业={pagerank_dict['中型企业'][i]:.4f}, 小型企业={pagerank_dict['小型企业'][i]:.4f}, 微型企业={pagerank_dict['微型企业'][i]:.4f}")

<Figure size 720x432 with 1 Axes>

2018-2019年PageRank值：
2018: 大型企业=0.1299, 中型企业=0.1202, 小型企业=0.1158, 微型企业=0.1185
2019: 大型企业=0.1416, 中型企业=0.1198, 小型企业=0.1134, 微型企业=0.1095

2022-2023年PageRank值：
2022: 大型企业=0.1370, 中型企业=0.1209, 小型企业=0.1152, 微型企业=0.1113
2023: 大型企业=0.1294, 中型企业=0.1212, 小型企业=0.1186, 微型企业=0.1152


In [9]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')

# 选择年份
year = 2023
df_year = df[df['年份'] == year]

# 提取各企业类型数据
large = df_year[df_year['企业规模'] == '大型企业'].iloc[0, 2:]
medium = df_year[df_year['企业规模'] == '中型企业'].iloc[0, 2:]
small = df_year[df_year['企业规模'] == '小型企业'].iloc[0, 2:]
micro = df_year[df_year['企业规模'] == '微型企业'].iloc[0, 2:]

# 构建权重大于0.2的新网络
G2 = nx.Graph()
G2.add_node('大型企业', type='企业')
G2.add_node('中型企业', type='企业')
G2.add_node('小型企业', type='企业')
G2.add_node('微型企业', type='企业')

partners = df.columns[2:]

for p in partners:
    G2.add_node(p, type='合作对象')
    if large[p] > 20:
        G2.add_edge('大型企业', p, weight=large[p])
    if medium[p] >20:
        G2.add_edge('中型企业', p, weight=medium[p])
    if small[p] > 20:
        G2.add_edge('小型企业', p, weight=small[p])
    if micro[p] > 20:
        G2.add_edge('微型企业', p, weight=micro[p])

# 自动布局
pos = nx.spring_layout(G2, seed=42)
edge_weights = [G2[u][v]['weight']/5 for u,v in G2.edges()]

# 节点颜色
node_colors = []
for node in G2.nodes():
    if node == '大型企业':
        node_colors.append('pink')
    elif node == '中型企业':
        node_colors.append('orange')
    elif node == '小型企业':
        node_colors.append('green')
    elif node == '微型企业':
        node_colors.append('cyan')
    else:
        node_colors.append('lightblue')

# 边颜色
brown_partners = ['高等学校', '研究机构', '政府部门或行业协会', '集团内其他企业']
black_partners = [
    '供应商', '客户', '竞争对手或同行业企业',
    '咨询顾问，市场分析及中介机构'
]


# 计算度中心性
degree_centrality = nx.degree_centrality(G2)
print(f"{year}年社交网络各节点度中心性：")
for node in ['大型企业', '中型企业', '小型企业', '微型企业']:
    print(f"{node}: {degree_centrality.get(node, 0):.4f}")

# 计算中介中心性
betweenness = nx.betweenness_centrality(G2, weight='weight')
print(f"\n{year}年社交网络各节点中介中心性：")
for node in ['大型企业', '中型企业', '小型企业', '微型企业']:
    print(f"{node}: {betweenness.get(node, 0):.4f}")

# 计算接近中心性
closeness = nx.closeness_centrality(G2, distance=lambda u, v, d: 1/d['weight'] if d['weight'] > 0 else float('inf'))
print(f"\n{year}年社交网络各节点接近中心性：")
for node in ['大型企业', '中型企业', '小型企业', '微型企业']:
    print(f"{node}: {closeness.get(node, 0):.4f}")

# 整合为DataFrame
closeness_df = pd.DataFrame({
    '年份': [year] * 4,
    '企业规模': ['大型企业', '中型企业', '小型企业', '微型企业'],
    '接近中心性': [closeness.get(node, 0) for node in ['大型企业', '中型企业', '小型企业', '微型企业']]
})
print("\n2023年各企业接近中心性：")
print(closeness_df)

2023年社交网络各节点度中心性：
大型企业: 0.5000
中型企业: 0.3333
小型企业: 0.3333
微型企业: 0.3333

2023年社交网络各节点中介中心性：
大型企业: 0.2727
中型企业: 0.0303
小型企业: 0.0606
微型企业: 0.1515

2023年社交网络各节点接近中心性：
大型企业: 17.5717
中型企业: 15.5225
小型企业: 14.5058
微型企业: 14.4424

2023年各企业接近中心性：
     年份  企业规模      接近中心性
0  2023  大型企业  17.571741
1  2023  中型企业  15.522526
2  2023  小型企业  14.505803
3  2023  微型企业  14.442442


In [10]:
import matplotlib.pyplot as plt

# 计算度中心性
degree_centrality = nx.degree_centrality(G2)
# 计算中介中心性
betweenness = nx.betweenness_centrality(G2, weight='weight')
# 计算接近中心性
closeness = nx.closeness_centrality(G2, distance=lambda u, v, d: 1/d['weight'] if d['weight'] > 0 else float('inf'))

# 合并为一个DataFrame
centrality_df = pd.DataFrame({
    '企业规模': ['大型企业', '中型企业', '小型企业', '微型企业'],
    '度中心性': [degree_centrality.get(node, 0) for node in ['大型企业', '中型企业', '小型企业', '微型企业']],
    '中介中心性': [betweenness.get(node, 0) for node in ['大型企业', '中型企业', '小型企业', '微型企业']],
    '接近中心性': [closeness.get(node, 0) for node in ['大型企业', '中型企业', '小型企业', '微型企业']]
})
print(f"\n{year}年各企业中心性指标：")
print(centrality_df)



fig, ax = plt.subplots(figsize=(7, 2 + 0.5*len(centrality_df)))
ax.axis('off')
tbl = ax.table(
    cellText=centrality_df.values,
    colLabels=centrality_df.columns,
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1.2, 1.2)

for row in range(len(centrality_df)+1):  
    tbl[(row, 0)].set_width(0.22)
plt.tight_layout()
plt.show()


2023年各企业中心性指标：
   企业规模      度中心性     中介中心性      接近中心性
0  大型企业  0.500000  0.272727  17.571741
1  中型企业  0.333333  0.030303  15.522526
2  小型企业  0.333333  0.060606  14.505803
3  微型企业  0.333333  0.151515  14.442442


<Figure size 504x288 with 1 Axes>

In [11]:
import matplotlib.pyplot as plt
import numpy as np

# 取消标准化，直接调整中介中心性和接近中心性
centrality_plot_df = centrality_df.copy()
centrality_plot_df['中介中心性'] = centrality_plot_df['中介中心性'] * 1.83
centrality_plot_df['接近中心性'] = centrality_plot_df['接近中心性'] / 35

labels = ['度中心性', '中介中心性', '接近中心性']
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

plt.figure(figsize=(7, 7))

for i, row in centrality_plot_df.iterrows():
    values = row[labels].tolist()
    values += values[:1]
    plt.polar(angles, values, marker='o', label=row['企业规模'])

plt.xticks(angles[:-1], labels, fontsize=13)
plt.yticks(fontsize=11)
plt.title('2023年各企业中心性指标雷达图（调整后）', fontsize=15)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.05))
plt.tight_layout()
plt.show()

<Figure size 504x504 with 1 Axes>